# Módulo 2 · Machine Learning Clásico: Supervisado, No Supervisado y Semi-supervisado

## Objetivos
1. Construir **pipelines reproducibles** en scikit-learn 1.x (`ColumnTransformer`, `Pipeline`, `set_output(transform="pandas")`).
2. Entender el **compromiso sesgo–varianza** y usarlo para elegir entre modelos lineales regularizados, árboles y *gradient boosting*.
3. Evaluar con la métrica correcta (ROC-AUC vs PR-AUC, calibración, *log-loss*) y con **validación cruzada bien hecha** (sin fuga de datos).
4. Aplicar aprendizaje **no supervisado**: K-means, GMM, DBSCAN, reducción de dimensionalidad (PCA, t-SNE/UMAP).
5. Aprovechar datos no etiquetados con **semi-supervisado** (*self-training*, *label spreading*).
6. Cuantificar la incertidumbre con **predicción conforme**, la técnica de calibración libre de distribución que se ha vuelto estándar en producción.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Qué es el aprendizaje estadístico, sesgo-varianza | James, Witten, Hastie, Tibshirani & Taylor, *ISLP* (2023) | **Cap. 2** Statistical Learning |
| Regresión lineal, clasificación (logística, LDA, KNN) | ISLP | **Cap. 3** Linear Regression · **Cap. 4** Classification |
| Validación cruzada y bootstrap | ISLP | **Cap. 5** Resampling Methods |
| Ridge, Lasso, selección de modelos | ISLP | **Cap. 6** Linear Model Selection and Regularization |
| Árboles, bagging, random forest, boosting | ISLP | **Cap. 8** Tree-Based Methods |
| Clustering y PCA | ISLP | **Cap. 12** Unsupervised Learning |
| Proyecto de ML de punta a punta, pipelines | Géron, *Hands-On ML* (3.ª ed., 2022) | **Cap. 2** End-to-End Machine Learning Project |
| Métricas de clasificación, curvas ROC/PR | Géron | **Cap. 3** Classification |
| Modelos lineales, SVM, árboles, ensambles | Géron | **Cap. 4-7** |
| Reducción de dimensionalidad y no supervisado | Géron | **Cap. 8** Dimensionality Reduction · **Cap. 9** Unsupervised Learning |
| Regresión, clasificación y ML estadístico con Python | Bruce et al. | **Cap. 4** Regression and Prediction · **Cap. 5** Classification · **Cap. 6** Statistical Machine Learning · **Cap. 7** Unsupervised Learning |

> **Nota de actualización (2026).** Se sustituye la edición en R de ISL (2021) por **ISLP** (edición Python, 2023, con la librería `ISLP`). Se añade *gradient boosting* con histogramas (`HistGradientBoostingClassifier`, XGBoost, LightGBM, CatBoost), que domina las competiciones tabulares, y **predicción conforme**, ausente en las ediciones anteriores de los libros pero hoy estándar (ver *MAPIE*).

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 2 · Machine Learning Clásico**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay,
                             log_loss, brier_score_loss, classification_report, mean_squared_error, r2_score)
print("scikit-learn", sklearn.__version__)
rng = np.random.default_rng(0)

scikit-learn 1.9.1


## 1. El compromiso sesgo–varianza (ISLP Cap. 2)

Para un modelo $\hat f$ entrenado sobre datos con ruido $\epsilon$ ($\mathrm{Var}(\epsilon)=\sigma^2$), el error esperado en un punto $x_0$ se descompone:

$$\mathbb{E}\big[(y_0-\hat f(x_0))^2\big] = \underbrace{\big(\mathbb{E}[\hat f(x_0)] - f(x_0)\big)^2}_{\text{sesgo}^2} + \underbrace{\mathrm{Var}\big(\hat f(x_0)\big)}_{\text{varianza}} + \underbrace{\sigma^2}_{\text{irreducible}}.$$

Modelos flexibles (polinomios de alto grado, árboles profundos, redes grandes) tienen bajo sesgo y alta varianza; los rígidos, al revés. **Lo simulamos** entrenando polinomios de distinto grado sobre muchos conjuntos de datos.

In [2]:
f_true = lambda x: np.sin(2 * np.pi * x)
x_grid = np.linspace(0, 1, 200)[:, None]

def experimento(grado, n_rep=200, n=30, sigma=0.3):
    preds = []
    for _ in range(n_rep):
        x = rng.uniform(0, 1, size=(n, 1)); y = f_true(x[:, 0]) + rng.normal(0, sigma, n)
        modelo = make_pipeline(PolynomialFeatures(grado), LinearRegression()).fit(x, y)
        preds.append(modelo.predict(x_grid))
    P = np.array(preds)
    sesgo2 = ((P.mean(0) - f_true(x_grid[:, 0])) ** 2).mean()
    var = P.var(0).mean()
    return sesgo2, var, P

grados = [1, 2, 3, 5, 9, 15]
res = [experimento(g) for g in grados]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for g, (_, _, P) in zip([1, 3, 15], [res[0], res[2], res[5]]):
    axes[0].plot(x_grid, P[:15].T, color=f"C{[1,3,15].index(g)}", alpha=0.25)
    axes[0].plot([], [], color=f"C{[1,3,15].index(g)}", label=f"grado {g}")
axes[0].plot(x_grid, f_true(x_grid[:, 0]), "k", lw=2, label="f verdadera"); axes[0].set_ylim(-2, 2); axes[0].legend()
axes[0].set_title("15 ajustes por grado: baja varianza vs alta varianza")
axes[1].plot(grados, [r[0] for r in res], "o-", label="sesgo²"); axes[1].plot(grados, [r[1] for r in res], "s-", label="varianza")
axes[1].plot(grados, [r[0] + r[1] + 0.3**2 for r in res], "k^-", label="error total esperado")
axes[1].set_xlabel("grado del polinomio"); axes[1].set_yscale("log"); axes[1].legend(); axes[1].set_title("Descomposición sesgo-varianza")
plt.show()

## 2. Un proyecto de punta a punta con `Pipeline` (Géron Cap. 2)

Regla de oro: **todo el preprocesamiento va dentro del `Pipeline`**. Así la validación cruzada re-ajusta imputadores, escaladores y codificadores *solo con el fold de entrenamiento* y evitamos la fuga de datos (*data leakage*), el error más común en ML aplicado.

Usaremos un dataset sintético "realista" de riesgo de crédito con variables numéricas, categóricas y valores faltantes.

In [3]:
def generar_credito(n=6000, seed=1):
    r = np.random.default_rng(seed)
    edad = r.integers(21, 70, n)
    ingreso = r.lognormal(10.3, 0.5, n)
    deuda_ratio = np.clip(r.beta(2, 5, n) * 1.5, 0, 1.5)
    antiguedad = r.integers(0, 30, n)
    empleo = r.choice(["asalariado", "independiente", "informal", "jubilado"], n, p=[0.55, 0.2, 0.15, 0.1])
    region = r.choice(["norte", "sur", "este", "oeste"], n)
    logit = (-2.2 + 2.8 * deuda_ratio - 0.00004 * (ingreso - 30000) - 0.03 * antiguedad
             + 0.6 * (empleo == "informal") + 0.3 * (empleo == "independiente") + 0.01 * (45 - edad))
    y = r.binomial(1, 1 / (1 + np.exp(-logit)))
    df = pd.DataFrame(dict(edad=edad, ingreso=ingreso.round(0), deuda_ratio=deuda_ratio.round(3),
                           antiguedad=antiguedad, empleo=empleo, region=region, default=y))
    # valores faltantes realistas
    df.loc[r.random(n) < 0.08, "ingreso"] = np.nan
    df.loc[r.random(n) < 0.03, "empleo"] = np.nan
    return df

df = generar_credito()
X, y = df.drop(columns="default"), df["default"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print("tasa de default:", y.mean().round(3)); display(df.head())

num_cols = ["edad", "ingreso", "deuda_ratio", "antiguedad"]
cat_cols = ["empleo", "region"]
preproc = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat_cols),
]).set_output(transform="pandas")   # novedad sklearn>=1.2: salida como DataFrame con nombres

display(preproc.fit_transform(X_tr).head(3))

tasa de default: 0.233


,edad,ingreso,deuda_ratio,antiguedad,empleo,region,default
0,44,94655.0,0.730,22,asalariado,oeste,0
1,46,28776.0,0.507,20,asalariado,sur,0
2,58,38123.0,0.509,9,asalariado,este,1
3,67,19954.0,0.385,19,independiente,norte,0
4,22,38567.0,0.104,14,jubilado,sur,0


,num__edad,num__ingreso,num__deuda_ratio,num__antiguedad,cat__empleo_asalariado,cat__empleo_independiente,cat__empleo_informal,cat__empleo_jubilado,cat__region_este,cat__region_norte,cat__region_oeste,cat__region_sur
1620,1.490888,-0.563446,0.057761,1.453887,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4771,1.349682,-0.508630,-0.756555,-1.323405,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2392,-0.909620,-0.209754,0.688335,-0.976244,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 3. Clasificación: de la regresión logística al *gradient boosting*

**Regresión logística** (ISLP 4.3): modela $\log\frac{p}{1-p} = \mathbf{w}^\top\mathbf{x}$ y se ajusta maximizando la verosimilitud (equivale a minimizar la *log-loss*). Es interpretable y bien calibrada: siempre debe ser tu *baseline*.

**Árboles y ensambles** (ISLP Cap. 8): un árbol particiona el espacio con reglas `si-entonces`; *bagging*/**random forest** promedia muchos árboles sobre remuestras para bajar la varianza; **boosting** ajusta árboles pequeños secuencialmente sobre los residuos (gradiente de la pérdida). `HistGradientBoostingClassifier` (inspirado en LightGBM) es nativo de sklearn, maneja `NaN` y categóricas, y es el punto de partida moderno para datos tabulares.

In [4]:
modelos = {
    "logística": LogisticRegression(max_iter=2000),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "hist gradient boosting": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=0),
}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
tabla = {}
for nombre, clf in modelos.items():
    pipe = Pipeline([("prep", preproc), ("clf", clf)])
    sc = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=["roc_auc", "average_precision", "neg_log_loss"], n_jobs=-1)
    tabla[nombre] = {k.replace("test_", ""): f"{v.mean():.3f} ± {v.std():.3f}" for k, v in sc.items() if k.startswith("test_")}
display(pd.DataFrame(tabla).T)

,roc_auc,average_precision,neg_log_loss
logística,0.737 ± 0.013,0.474 ± 0.012,-0.475 ± 0.008
random forest,0.727 ± 0.021,0.441 ± 0.021,-0.484 ± 0.011
hist gradient boosting,0.710 ± 0.018,0.419 ± 0.026,-0.526 ± 0.025


### Elegir la métrica correcta (Géron Cap. 3)
* **ROC-AUC**: probabilidad de rankear un positivo por encima de un negativo. Insensible al desbalance → puede verse "bien" aunque el modelo sea inútil para la clase rara.
* **PR-AUC / average precision**: se centra en la clase positiva; preferible cuando los positivos son escasos (fraude, enfermedad).
* **Log-loss / Brier**: evalúan la **calidad de las probabilidades**, no solo el orden. Cruciales si las probabilidades se usan para decidir (pricing, triage).
* La **exactitud** (*accuracy*) casi nunca es la métrica de negocio: con 10 % de positivos, predecir siempre "no" da 90 %.

In [5]:
pipe_gb = Pipeline([("prep", preproc), ("clf", modelos["hist gradient boosting"])]).fit(X_tr, y_tr)
pipe_lr = Pipeline([("prep", preproc), ("clf", modelos["logística"])]).fit(X_tr, y_tr)
p_gb, p_lr = pipe_gb.predict_proba(X_te)[:, 1], pipe_lr.predict_proba(X_te)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for p, nombre in [(p_lr, "logística"), (p_gb, "HGB")]:
    RocCurveDisplay.from_predictions(y_te, p, name=nombre, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_te, p, name=nombre, ax=axes[1])
    # curva de calibración manual
    bins = np.quantile(p, np.linspace(0, 1, 11)); idx = np.clip(np.digitize(p, bins) - 1, 0, 9)
    axes[2].plot([p[idx == b].mean() for b in range(10)], [y_te.values[idx == b].mean() for b in range(10)], "o-", label=nombre)
axes[2].plot([0, 1], [0, 1], "k--"); axes[2].set_xlabel("prob. predicha"); axes[2].set_ylabel("frecuencia observada")
axes[2].set_title("Calibración"); axes[2].legend(); axes[0].set_title("ROC"); axes[1].set_title("Precision-Recall"); plt.show()
print(f"Brier logística={brier_score_loss(y_te, p_lr):.4f}  HGB={brier_score_loss(y_te, p_gb):.4f}")

Brier logística=0.1559  HGB=0.1738


### Umbral de decisión = decisión de negocio
El umbral 0.5 es arbitrario. Si un *default* no detectado cuesta 10× más que rechazar a un buen cliente, el umbral óptimo sale de minimizar el **costo esperado**. En sklearn ≥ 1.5 existe `TunedThresholdClassifierCV` para hacerlo dentro de la validación cruzada.

In [6]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import make_scorer, confusion_matrix

def costo(y_true, y_pred, c_fn=10, c_fp=1):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return -(c_fn * fn + c_fp * fp)   # negativo porque sklearn maximiza

pipe_gb_umbral = TunedThresholdClassifierCV(pipe_gb, scoring=make_scorer(costo), cv=5).fit(X_tr, y_tr)
print(f"umbral óptimo por costo: {pipe_gb_umbral.best_threshold_:.3f}")
for nombre, pred in [("umbral 0.5", (p_gb >= 0.5).astype(int)), ("umbral ajustado", pipe_gb_umbral.predict(X_te))]:
    print(f"{nombre:16s} costo total = {-costo(y_te, pred):,}")

umbral óptimo por costo: 0.040
umbral 0.5       costo total = 2,861
umbral ajustado  costo total = 1,169


## 4. Regresión regularizada: Ridge y Lasso (ISLP Cap. 6)

$$\hat{\mathbf{w}}_{\text{ridge}} = \arg\min \|\mathbf{y}-X\mathbf{w}\|^2 + \lambda\|\mathbf{w}\|_2^2,\qquad
\hat{\mathbf{w}}_{\text{lasso}} = \arg\min \|\mathbf{y}-X\mathbf{w}\|^2 + \lambda\|\mathbf{w}\|_1 .$$

Ridge encoge todos los coeficientes; Lasso puede llevarlos exactamente a cero (selección de variables). Veremos las **trayectorias de regularización** y elegiremos $\lambda$ por validación cruzada.

In [7]:
from sklearn.linear_model import RidgeCV, LassoCV, lasso_path
n, p = 300, 20
Xr = rng.normal(size=(n, p)); w_true = np.zeros(p); w_true[:5] = [3, -2, 1.5, 0, 0.8]
yr = Xr @ w_true + rng.normal(0, 1.5, n)

alphas, coefs, _ = lasso_path(Xr, yr, alphas=np.logspace(0.5, -2.5, 60))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(alphas, coefs.T); axes[0].set_xscale("log"); axes[0].invert_xaxis(); axes[0].set_xlabel("λ (alpha)")
axes[0].set_title("Trayectoria Lasso: coeficientes entran uno a uno")
ridge_coefs = np.array([Ridge(alpha=a).fit(Xr, yr).coef_ for a in np.logspace(-2, 4, 60)])
axes[1].plot(np.logspace(-2, 4, 60), ridge_coefs); axes[1].set_xscale("log"); axes[1].set_xlabel("λ (alpha)")
axes[1].set_title("Trayectoria Ridge: encogimiento suave"); plt.show()

lasso = LassoCV(cv=5, random_state=0).fit(Xr, yr); ridge = RidgeCV(alphas=np.logspace(-2, 4, 60)).fit(Xr, yr)
print("Lasso λ*=%.3f  variables seleccionadas: %s" % (lasso.alpha_, np.flatnonzero(np.abs(lasso.coef_) > 1e-6)))
print("Ridge λ*=%.3f" % ridge.alpha_)

Lasso λ*=0.096  variables seleccionadas: [ 0  1  2  3  4  6  8 14 19]
Ridge λ*=2.759


## 5. Búsqueda de hiperparámetros y curvas de aprendizaje

* `RandomizedSearchCV` suele batir a `GridSearchCV` con el mismo presupuesto (Bergstra & Bengio, 2012).
* `HalvingRandomSearchCV` (*successive halving*) descarta configuraciones malas con pocos datos y concentra el presupuesto en las buenas.
* Para búsquedas serias en 2026: **Optuna** (bayesiana/TPE, *pruning*), integrada con sklearn, PyTorch y LightGBM.
* La **curva de aprendizaje** te dice si necesitas más datos (brecha train/val) o un modelo más flexible (ambos errores altos).

In [8]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import loguniform, randint

param_dist = {"clf__learning_rate": loguniform(0.01, 0.3), "clf__max_leaf_nodes": randint(8, 64),
              "clf__l2_regularization": loguniform(1e-3, 10), "clf__min_samples_leaf": randint(10, 100)}
search = HalvingRandomSearchCV(Pipeline([("prep", preproc), ("clf", HistGradientBoostingClassifier(max_iter=200, random_state=0))]),
                               param_dist, factor=3, resource="n_samples", min_resources=600,
                               scoring="neg_log_loss", cv=3, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print("mejores parámetros:", {k: round(v, 4) if isinstance(v, float) else v for k, v in search.best_params_.items()})
print("log-loss test:", round(log_loss(y_te, search.predict_proba(X_te)[:, 1]), 4))

sizes, tr_sc, va_sc = learning_curve(search.best_estimator_, X_tr, y_tr, cv=3, scoring="neg_log_loss",
                                     train_sizes=np.linspace(0.1, 1, 6), n_jobs=-1)
plt.plot(sizes, -tr_sc.mean(1), "o-", label="train"); plt.plot(sizes, -va_sc.mean(1), "s-", label="validación")
plt.xlabel("n muestras de entrenamiento"); plt.ylabel("log-loss"); plt.title("Curva de aprendizaje"); plt.legend(); plt.show()

mejores parámetros: {'clf__l2_regularization': np.float64(0.0155), 'clf__learning_rate': np.float64(0.0121), 'clf__max_leaf_nodes': 20, 'clf__min_samples_leaf': 68}


log-loss test: 0.4902


## 6. Aprendizaje no supervisado (ISLP Cap. 12; Géron Cap. 8-9)

Sin etiquetas buscamos **estructura**:
* **K-means**: minimiza la inercia $\sum_k\sum_{i\in C_k}\|x_i-\mu_k\|^2$; asume clusters esféricos y de tamaño similar. Elige $k$ con el *silhouette score*, no solo con el "codo".
* **Mezclas gaussianas (GMM)**: K-means probabilístico con covarianzas; permite clusters elípticos y da probabilidades de pertenencia. Selecciona $k$ con BIC.
* **DBSCAN / HDBSCAN**: basados en densidad; descubren formas arbitrarias y marcan ruido; no requieren $k$.
* **PCA** (lineal) vs **t-SNE / UMAP** (no lineales, solo para visualizar; ¡no interpretes distancias globales!).

In [9]:
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

Xb, _ = make_blobs(n_samples=600, centers=4, cluster_std=[1.0, 1.8, 0.6, 1.2], random_state=3)
Xm, _ = make_moons(n_samples=600, noise=0.07, random_state=0)

ks = range(2, 9)
sil = [silhouette_score(Xb, KMeans(k, n_init=10, random_state=0).fit_predict(Xb)) for k in ks]
bic = [GaussianMixture(k, n_init=3, random_state=0).fit(Xb).bic(Xb) for k in ks]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
axes[0].plot(ks, sil, "o-"); axes[0].set_title("silhouette (K-means) → k=%d" % ks[int(np.argmax(sil))]); axes[0].set_xlabel("k")
axes[1].plot(ks, bic, "s-"); axes[1].set_title("BIC (GMM) → k=%d" % ks[int(np.argmin(bic))]); axes[1].set_xlabel("k")
axes[2].scatter(*Xm.T, c=KMeans(2, n_init=10, random_state=0).fit_predict(Xm), s=8, cmap="coolwarm"); axes[2].set_title("K-means falla en 'lunas'")
axes[3].scatter(*Xm.T, c=DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm), s=8, cmap="coolwarm"); axes[3].set_title("DBSCAN (densidad) acierta")
plt.show()

In [10]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

digits = load_digits(); Xd, yd = digits.data, digits.target
pca = PCA(n_components=50, random_state=0).fit(Xd)
print("componentes para 90% de varianza:", np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9) + 1)
Z_pca = pca.transform(Xd)[:, :2]
Z_tsne = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(pca.transform(Xd))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, Z, t in [(axes[0], Z_pca, "PCA (lineal, global)"), (axes[1], Z_tsne, "t-SNE sobre 50 PCs (local, solo visual)")]:
    sc = ax.scatter(*Z.T, c=yd, cmap="tab10", s=6); ax.set_title(t)
plt.colorbar(sc, ax=axes, ticks=range(10), fraction=0.02); plt.show()

componentes para 90% de varianza: 21


## 7. Aprendizaje semi-supervisado

Escenario habitual: miles de registros, pocos etiquetados (etiquetar es caro). Dos familias en sklearn:
* **Self-training** (`SelfTrainingClassifier`): entrena con lo etiquetado, pseudo-etiqueta lo no etiquetado con alta confianza, repite.
* **Propagación de etiquetas** (`LabelSpreading`): construye un grafo de similitud y difunde las etiquetas por él — el mismo principio de los GNN (Módulo 7).

Convención: las muestras no etiquetadas llevan `-1`.

In [11]:
from sklearn.semi_supervised import SelfTrainingClassifier, LabelSpreading
from sklearn.svm import SVC

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, stratify=yd, random_state=0)
mask = rng.random(len(yd_tr)) < 0.05                      # solo 5% etiquetado (~63 dígitos)
y_semi = np.where(mask, yd_tr, -1)
print(f"etiquetados: {mask.sum()} de {len(yd_tr)}")

sup = make_pipeline(StandardScaler(), SVC(probability=True, gamma="scale")).fit(Xd_tr[mask], yd_tr[mask])
selftr = make_pipeline(StandardScaler(), SelfTrainingClassifier(SVC(probability=True, gamma="scale"), threshold=0.9)).fit(Xd_tr, y_semi)
ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2).fit(Xd_tr, y_semi)

for nombre, m in [("solo supervisado (5%)", sup), ("self-training", selftr), ("label spreading", ls)]:
    print(f"{nombre:24s} exactitud test = {m.score(Xd_te, yd_te):.3f}")

etiquetados: 80 de 1257


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


solo supervisado (5%)    exactitud test = 0.791
self-training            exactitud test = 0.833
label spreading          exactitud test = 0.946


## 8. Predicción conforme: intervalos con garantía

Dado un modelo *cualquiera* y un conjunto de **calibración** separado, la predicción conforme (*split conformal*) construye conjuntos $C(x)$ tales que
$$\Pr\big(y_{n+1}\in C(x_{n+1})\big)\ \ge\ 1-\alpha$$
**sin supuestos sobre la distribución** (solo intercambiabilidad). Receta para regresión:
1. Ajusta $\hat f$ en entrenamiento.
2. Calcula los residuos absolutos $s_i=|y_i-\hat f(x_i)|$ en calibración.
3. Toma $\hat q$ = cuantil $\lceil (n+1)(1-\alpha)\rceil/n$ de los $s_i$.
4. Intervalo: $[\hat f(x)-\hat q,\ \hat f(x)+\hat q]$.

En producción usa **MAPIE** (`pip install mapie`) o `crepes`; aquí lo implementamos a mano para entenderlo (Angelopoulos & Bates, 2023).

In [12]:
from sklearn.ensemble import HistGradientBoostingRegressor
xr = rng.uniform(-3, 3, 3000); yr = np.sin(xr) * 2 + rng.normal(0, 0.3 + 0.3 * np.abs(xr), 3000)  # ruido heterocedástico
X_fit, X_cal, X_test = xr[:1500, None], xr[1500:2250, None], xr[2250:, None]
y_fit, y_cal, y_test = yr[:1500], yr[1500:2250], yr[2250:]

alpha = 0.1
reg = HistGradientBoostingRegressor(random_state=0).fit(X_fit, y_fit)
scores = np.abs(y_cal - reg.predict(X_cal))
n_cal = len(scores); q = np.quantile(scores, np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, method="higher")
pred = reg.predict(X_test); cobertura = ((y_test >= pred - q) & (y_test <= pred + q)).mean()
print(f"q̂ = {q:.3f}   cobertura empírica en test = {cobertura:.3f}  (objetivo ≥ {1-alpha})")

# Versión 'adaptativa': normaliza el score por una estimación de la dispersión -> intervalos que se ensanchan con |x|
disp = HistGradientBoostingRegressor(random_state=0).fit(X_fit, np.abs(y_fit - reg.predict(X_fit)))
scores_n = np.abs(y_cal - reg.predict(X_cal)) / disp.predict(X_cal)
q_n = np.quantile(scores_n, np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, method="higher")
half = q_n * disp.predict(X_test); cob_n = ((y_test >= pred - half) & (y_test <= pred + half)).mean()
print(f"adaptativo: cobertura = {cob_n:.3f}, ancho medio {2*half.mean():.2f} vs fijo {2*q:.2f}")

o = np.argsort(X_test[:, 0])
plt.figure(figsize=(9, 4)); plt.scatter(X_test, y_test, s=5, alpha=0.4, color="gray")
plt.fill_between(X_test[o, 0], pred[o] - q, pred[o] + q, alpha=0.25, label="conforme fijo")
plt.fill_between(X_test[o, 0], pred[o] - half[o], pred[o] + half[o], alpha=0.25, label="conforme adaptativo")
plt.plot(X_test[o, 0], pred[o], "k", lw=1); plt.legend(); plt.title("Intervalos con cobertura garantizada del 90%"); plt.show()

q̂ = 1.416   cobertura empírica en test = 0.919  (objetivo ≥ 0.9)


adaptativo: cobertura = 0.901, ancho medio 2.75 vs fijo 2.83


## 📦 Recursos de los repositorios oficiales de los libros

| Libro | Recurso | Para qué usarlo aquí |
|---|---|---|
| James et al., *ISLP* | Labs oficiales: [`Ch02-statlearn-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch02-statlearn-lab.ipynb), [`Ch03-linreg-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch03-linreg-lab.ipynb), [`Ch04-classification-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch04-classification-lab.ipynb), [`Ch05-resample-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch05-resample-lab.ipynb), [`Ch06-varselect-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch06-varselect-lab.ipynb), [`Ch08-baggboost-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch08-baggboost-lab.ipynb), [`Ch12-unsup-lab`](https://github.com/intro-stat-learning/ISLP_labs/blob/main/Ch12-unsup-lab.ipynb). Instalación: `uv pip install -r https://raw.githubusercontent.com/intro-stat-learning/ISLP_labs/v2.2.2/requirements.txt` | Los labs usan `statsmodels` + la librería `ISLP` (datasets `Boston`, `Default`, `Hitters`, `NCI60`): complementan la visión de *inferencia* que sklearn no da (p-valores, intervalos) |
| Géron, *Hands-On ML 3* | [`02_end_to_end_machine_learning_project.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb) (California housing, pipelines), [`03_classification.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/03_classification.ipynb) (MNIST, métricas), [`04_training_linear_models.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/04_training_linear_models.ipynb), [`06_decision_trees.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/06_decision_trees.ipynb), [`07_ensemble_learning_and_random_forests.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/07_ensemble_learning_and_random_forests.ipynb), [`08_dimensionality_reduction.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/08_dimensionality_reduction.ipynb), [`09_unsupervised_learning.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/09_unsupervised_learning.ipynb) | El Cap. 2 es **el** proyecto guiado de referencia; hazlo completo tras la sección 2. El Cap. 9 incluye semi-supervisado con clustering (propagación de etiquetas sobre `digits`, como nuestra sección 7) |
| Bruce et al., *Practical Statistics* | [`Chapter 4 - Regression and Prediction`](https://github.com/gedeck/practical-statistics-for-data-scientists/blob/master/python/notebooks/Chapter%204%20-%20Regression%20and%20Prediction.ipynb), [`Chapter 5 - Classification`](https://github.com/gedeck/practical-statistics-for-data-scientists/blob/master/python/notebooks/Chapter%205%20-%20Classification.ipynb), [`Chapter 6 - Statistical Machine Learning`](https://github.com/gedeck/practical-statistics-for-data-scientists/blob/master/python/notebooks/Chapter%206%20-%20Statistical%20Machine%20Learning.ipynb), [`Chapter 7 - Unsupervised Learning`](https://github.com/gedeck/practical-statistics-for-data-scientists/blob/master/python/notebooks/Chapter%207%20-%20Unsupervised%20Learning.ipynb) · datasets [`loan_data.csv.gz`, `lc_loans.csv`, `house_sales.csv`, `sp500_data.csv.gz`](https://github.com/gedeck/practical-statistics-for-data-scientists/tree/master/data) | `loan_data.csv.gz` (Lending Club) es un dataset real de crédito para repetir la sección 3 con datos no sintéticos |
| Zhang et al., *D2L* | [Cap. 19 Hyperparameter Optimization](https://d2l.ai/chapter_hyperparameter-optimization/index.html) (búsqueda aleatoria, *successive halving*, Hyperband) | Profundiza la sección 5 con la teoría de los métodos multi-fidelidad |

## 9. Estado de la práctica en 2026 (para saber qué estudiar después)

* **Tabular:** *gradient boosting* (LightGBM/XGBoost/CatBoost) sigue siendo el rey; los *foundation models* tabulares (TabPFN v2, 2025) ya compiten en conjuntos pequeños/medianos sin entrenar. Evalúa ambos.
* **AutoML:** AutoGluon, FLAML y `sklearn` + Optuna. Útil como *baseline* fuerte, no como caja negra final.
* **Datos faltantes y categóricas:** los boosters modernos los tratan nativamente; el *one-hot* masivo es cosa del pasado (usa `TargetEncoder` de sklearn ≥ 1.3 o codificación nativa).
* **Fuga de datos temporal:** en series de tiempo usa `TimeSeriesSplit`; nunca `shuffle=True`.
* **Interpretabilidad:** SHAP, importancias por permutación y dependencia parcial (Módulo 10).

## 10. Ejercicios
1. Añade `XGBoost` o `LightGBM` (si están instalados) al comparativo de la sección 3 y compara tiempo de entrenamiento y log-loss.
2. Introduce deliberadamente **fuga de datos**: escala `X` completo *antes* del `train_test_split` y compara la estimación de CV con el test. Luego hazlo con una variable "futura" (p. ej. `mora_30d` derivada de `default`). Documenta la diferencia.
3. Implementa **Elastic Net** con `ElasticNetCV` sobre los datos de la sección 4 y compara el número de variables seleccionadas con Lasso.
4. Ejecuta `HDBSCAN` (sklearn ≥ 1.3) sobre `make_moons` con ruido 0.12 y compara con DBSCAN. ¿Cuál requiere menos ajuste?
5. En el ejemplo semi-supervisado, varía la fracción etiquetada (1 %, 5 %, 20 %, 100 %) y grafica la exactitud de cada método. ¿Cuándo deja de ayudar el no etiquetado?
6. Extiende la predicción conforme a **clasificación**: usa el score $1-\hat p_y(x)$ en calibración y construye conjuntos de etiquetas para `digits`. Mide cobertura y tamaño medio del conjunto.

## Referencias
* James, G., Witten, D., Hastie, T., Tibshirani, R. & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. Cap. 2-6, 8, 12. Labs: github.com/intro-stat-learning/ISLP_labs
* Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3.ª ed.). O'Reilly. Cap. 2-9. Código: github.com/ageron/handson-ml3
* Bruce, P., Bruce, A. & Gedeck, P. (2020). *Practical Statistics for Data Scientists* (2.ª ed.). Cap. 4-7.
* Angelopoulos, A. & Bates, S. (2023). *Conformal Prediction: A Gentle Introduction*. Foundations and Trends in ML.
* Hollmann, N. et al. (2025). *Accurate predictions on small data with a tabular foundation model* (TabPFN). Nature.
* Documentación scikit-learn 1.9: https://scikit-learn.org/stable/